# Reading the Data

In [1]:
import os
import glob
import zipfile
import json
import pandas as pd

In [2]:
# Define the zip file pattern
zip_pattern = "data/experiment_data_P0*.zip"

# Lists to store the parsed data
all_csv_trials = []
all_clicks = []
all_mouse_samples = []
raw_json_data = []

# Iterate through all zip files matching the pattern
for zip_path in glob.glob(zip_pattern):
    if os.path.exists(zip_path):
        print(f"Processing {zip_path}...")
        with zipfile.ZipFile(zip_path, 'r') as z:
            # Identify the specific JSON and CSV files inside this zip
            file_names = z.namelist()
            json_filename = next((f for f in file_names if f.endswith('.json')), None)
            csv_filename = next((f for f in file_names if f.endswith('.csv')), None)

            # Read and process the CSV file (trial summary)
            if csv_filename:
                with z.open(csv_filename) as csv_file:
                    df_csv = pd.read_csv(csv_file)
                    all_csv_trials.append(df_csv)

            # Read and process the JSON file
            if json_filename:
                with z.open(json_filename) as json_file:
                    data = json.loads(json_file.read().decode('utf-8'))
                    raw_json_data.append(data)

                    p_code = data.get('participant_code', 'Unknown')

                    # Flatten click_events
                    if 'click_events' in data:
                        for click in data['click_events']:
                            click['participant_code'] = p_code
                            all_clicks.append(click)

                    # Flatten mouse_samples
                    if 'mouse_samples' in data:
                        for sample in data['mouse_samples']:
                            sample['participant_code'] = p_code
                            all_mouse_samples.append(sample)

print(f"Successfully processed {len(raw_json_data)} participant zip files.")

Processing data\experiment_data_P001.zip...
Processing data\experiment_data_P002.zip...
Processing data\experiment_data_P003.zip...
Processing data\experiment_data_P004.zip...
Processing data\experiment_data_P005.zip...
Processing data\experiment_data_P006.zip...
Processing data\experiment_data_P007.zip...
Processing data\experiment_data_P008.zip...
Processing data\experiment_data_P009.zip...
Processing data\experiment_data_P010.zip...
Processing data\experiment_data_P011.zip...
Processing data\experiment_data_P012.zip...
Processing data\experiment_data_P013.zip...
Processing data\experiment_data_P014.zip...
Processing data\experiment_data_P015.zip...
Processing data\experiment_data_P016.zip...
Processing data\experiment_data_P017.zip...
Processing data\experiment_data_P018.zip...
Processing data\experiment_data_P019.zip...
Processing data\experiment_data_P020.zip...
Successfully processed 20 participant zip files.


In [7]:
# Combine everything into master pandas DataFrames
if all_csv_trials:
    df_trials = pd.concat(all_csv_trials, ignore_index=True)
else:
    df_trials = pd.DataFrame()

df_clicks = pd.DataFrame(all_clicks)
df_mouse = pd.DataFrame(all_mouse_samples)

print(f"Created df_trials with {len(df_trials)} rows from the CSVs.")
print(f"Created df_clicks with {len(df_clicks)} rows from the JSONs.")
print(f"Created df_mouse with {len(df_mouse)} rows from the JSONs.")

Created df_trials with 240 rows from the CSVs.
Created df_clicks with 1520 rows from the JSONs.
Created df_mouse with 301332 rows from the JSONs.


In [8]:
df_trials.head()

,participant_number,participant_id,participant_code,session_id,assigned_condition,participant_yes_button_side,participant_no_button_side,response_side_assignment_method,presentation_order_method,counterbalance_index,...,total_mouse_distance_confidence1,total_mouse_distance_phase3,total_mouse_distance_confidence2,number_of_pauses_phase1,number_of_pauses_phase3,number_of_direction_changes_phase1,number_of_direction_changes_phase3,dwell_time_yes_button_ms,dwell_time_no_button_ms,dwell_time_confidence_scale_ms
0,1,P_20260622185812_9ijiw,P001,S_20260622185812_gx8xgfxq,soft,left,right,fixed_by_participant_code,latin_square_counterbalancing,0,...,696,379,561,2,2,0,0,865,0,1449
1,1,P_20260622185812_9ijiw,P001,S_20260622185812_gx8xgfxq,soft,left,right,fixed_by_participant_code,latin_square_counterbalancing,0,...,519,341,644,4,4,0,0,1486,547,1351
2,1,P_20260622185812_9ijiw,P001,S_20260622185812_gx8xgfxq,soft,left,right,fixed_by_participant_code,latin_square_counterbalancing,0,...,581,433,927,2,2,0,0,2881,0,1900
3,1,P_20260622185812_9ijiw,P001,S_20260622185812_gx8xgfxq,soft,left,right,fixed_by_participant_code,latin_square_counterbalancing,0,...,461,478,492,3,3,5,0,602,511,1326
4,1,P_20260622185812_9ijiw,P001,S_20260622185812_gx8xgfxq,soft,left,right,fixed_by_participant_code,latin_square_counterbalancing,0,...,572,887,679,2,5,0,1,1359,483,2300


In [9]:
df_clicks.head()

,participant_number,participant_id,participant_code,session_id,assigned_condition,trial_index,problem_id,phase,clicked_element,click_timestamp_ms,x,y
0,1,P_20260622185812_9ijiw,P001,S_20260622185812_gx8xgfxq,soft,1,T02,trial_start,start_trial_button,125945,613,399
1,1,P_20260622185812_9ijiw,P001,S_20260622185812_gx8xgfxq,soft,1,T02,first_answer,yes_button,130033,229,575
2,1,P_20260622185812_9ijiw,P001,S_20260622185812_gx8xgfxq,soft,1,T02,confidence1,confidence_scale,132603,723,395
3,1,P_20260622185812_9ijiw,P001,S_20260622185812_gx8xgfxq,soft,1,T02,center_before_final,center_continue_button,136131,640,392
4,1,P_20260622185812_9ijiw,P001,S_20260622185812_gx8xgfxq,soft,1,T02,final_answer,yes_button,149957,341,546


In [10]:
df_mouse.head()

,participant_number,participant_id,participant_code,session_id,assigned_condition,trial_index,problem_id,phase,timestamp_ms_from_experiment_start,timestamp_ms_from_phase_start,x,y,viewport_width,viewport_height,element_or_zone,countdown_time_remaining_ms,mouse_button_down
0,1,P_20260622185812_9ijiw,P001,S_20260622185812_gx8xgfxq,soft,1,T02,first_answer,125972,26,613,399,1163,654,problem_text,9974,False
1,1,P_20260622185812_9ijiw,P001,S_20260622185812_gx8xgfxq,soft,1,T02,first_answer,125996,51,613,399,1163,654,problem_text,9950,False
2,1,P_20260622185812_9ijiw,P001,S_20260622185812_gx8xgfxq,soft,1,T02,first_answer,126023,77,613,399,1163,654,problem_text,9923,False
3,1,P_20260622185812_9ijiw,P001,S_20260622185812_gx8xgfxq,soft,1,T02,first_answer,126046,101,613,399,1163,654,problem_text,9899,False
4,1,P_20260622185812_9ijiw,P001,S_20260622185812_gx8xgfxq,soft,1,T02,first_answer,126071,126,613,399,1163,654,problem_text,9874,False
